# 1) Spin-the-Wheel Study Task Picker

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

Help a student start quickly by randomly picking one 10-minute study task and showing a simple challenge.

## Simple rules (policy)

- Pick one task per loop.
- Give a 10-minute challenge.
- Track pick counts in memory (JSON).

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [ ]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [ ]:
MEMORY_FILE = "01_spin_wheel_memory.json"

TASKS_DEFAULT = [
    "Revise last lecture notes",
    "Solve 2 easy problems",
    "Rewrite one concept in your own words",
    "Make 5 flashcards",
    "Refactor one function name + variables",
]

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action("terminate", {})

    tasks = memory.get("tasks") or TASKS_DEFAULT
    memory["tasks"] = tasks

    picks = memory.get("picks", {})
    choice = random.choice(tasks)
    picks[choice] = int(picks.get(choice, 0)) + 1
    memory["picks"] = picks

    msg = (
        f"Your task wheel picked: {choice}\n"
        "Challenge: do it for 10 minutes (no switching).\n"
        f"Times picked: {picks[choice]}"
    )
    return Action("notify", {"message": msg})

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}

    Tools.notify("Spin-the-Wheel Task Picker is running.")
    Tools.notify("Press Enter to spin. Type 'stop' to end.")

    user_text = Tools.ask("Ready?")
    while True:
        action = decide_next_action(user_text, memory)
        out = env.execute(action, memory)
        memory = out["memory"]

        if out["result"] == "terminated":
            env.execute(Action("save_memory", {}), memory)
            Tools.notify("Saved stats. Bye!")
            break

        env.execute(Action("save_memory", {}), memory)
        user_text = Tools.ask("Spin again? (Enter / stop)")

run_agent()


## Easy extensions

- Let students add/remove tasks during runtime.
- Add difficulty tags (easy/medium/hard).
- Print a small leaderboard of most-picked tasks.